In [2]:
# SET- UP ENVIRONMENT

!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers

In [3]:
# IMPORT LIBRARIES

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from pypdf import PdfReader
from transformers import pipeline

In [4]:
# LOAD EMBEDDING MODEL

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# LOAD PDF

from google.colab import files
uploaded = files.upload()

Saving IJSDR2506004.pdf to IJSDR2506004.pdf


In [6]:
pdf_path = list(uploaded.keys())[0]
reader = PdfReader(pdf_path)

text = ""
for page in reader.pages:
    text += page.extract_text()

In [7]:
with open(pdf_path, "rb") as f:
    print(f.read(10))

b'%PDF-1.5\r\n'


In [8]:
# CHUNKING

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text)
print("Number of chunks:", len(chunks))

Number of chunks: 39


In [9]:
# CREATE EMBEDDINGS

embeddings = embedding_model.encode(chunks)
embeddings = np.array(embeddings)

In [10]:
embeddings.shape

(39, 384)

In [11]:
# STORE IN FAISS

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [12]:
# LETS Ask Question

query = "What does the paper say about RECOMMENDATION SYSTEM?"
query_embedding = embedding_model.encode([query])

D, I = index.search(np.array(query_embedding), k=3)

In [13]:
retrieved_chunks = [chunks[i] for i in I[0]]
context = "\n".join(retrieved_chunks)

print(context)

allowing users to purchase books instantly. [4] 
The proposed recommendation system considers the number of 
users who have rated the books, rather than relying on the 
overall rating. Due to this, a recommendation might arise from 
a book that a user has given low rating to, in which case a book 
might be recommended from a genre that the user dislikes. This 
recommendation system relies on the ratings given by users. So, 
trust is a major issue, like whether the feedback and rating given 
by t
 
recommendation system takes into account various 
factors such  as ratings, book titles, prices, and more.  
Machine learning has been  enhancing the 
recommendation system and also opens up additional 
opportunities to boost its performance. [1] 
 
8. REFERENCES 
[1] BACHHAV, A., UKIRADE, A., PATIL, N., 
SASWADKAR, M., & SHIVALE, N. (2022). Book 
recommendation system using machine learning and 
collaborative filtering.  International Journal of Advanced 
Research in Science, Communication a

In [14]:
# Add LLM for Final Answer

generator = pipeline("text-generation", model="google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [15]:
prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = generator(prompt, max_length=512)
print(response[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using ONLY the context below.

Context:
allowing users to purchase books instantly. [4] 
The proposed recommendation system considers the number of 
users who have rated the books, rather than relying on the 
overall rating. Due to this, a recommendation might arise from 
a book that a user has given low rating to, in which case a book 
might be recommended from a genre that the user dislikes. This 
recommendation system relies on the ratings given by users. So, 
trust is a major issue, like whether the feedback and rating given 
by t
 
recommendation system takes into account various 
factors such  as ratings, book titles, prices, and more.  
Machine learning has been  enhancing the 
recommendation system and also opens up additional 
opportunities to boost its performance. [1] 
 
8. REFERENCES 
[1] BACHHAV, A., UKIRADE, A., PATIL, N., 
SASWADKAR, M., & SHIVALE, N. (2022). Book 
recommendation system using machine learning and 
collaborative filtering.  Internatio